In [1]:
import pipelines
import pandas as pd
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('hotel_bookings.csv')

preparation = pipelines.default_data_preparation_pipeline
preparation.fit(df, None)
tmp_df = preparation.transform(df)


In [2]:
from ml_utils import run_experiment
from sklearn.pipeline import Pipeline
log_reg_pipe = Pipeline(
    [
        ('Data Preparation', pipelines.default_data_preparation_pipeline),
        ('model', LogisticRegression())
    ]
)
log_reg_pipe

Pipeline(steps=[('Data Preparation',
                 Pipeline(steps=[('Feature Engineering', FeatureEngineer()),
                                 ('feature encoding',
                                  ColumnTransformer(transformers=[('categorial_oh',
                                                                   OneHotEncoder(handle_unknown='ignore'),
                                                                   ['hotel',
                                                                    'arrival_date_month',
                                                                    'meal',
                                                                    'market_segment',
                                                                    'distribution_channel',
                                                                    'reserved_room_type',
                                                                    'deposit_type',
                                                                    'customer_type',
                                                                    'country_group']),
                                                                  ('nu...
                                                                    'adults',
                                                                    'children',
                                                                    'babies',
                                                                    'is_repeated_guest',
                                                                    'previous_cancellations',
                                                                    'previous_bookings_not_canceled',
                                                                    'booking_changes',
                                                                    'days_in_waiting_list',
                                                                    'required_car_parking_spaces',
                                                                    'total_of_special_requests',
                                                                    'total_nights',
                                                                    'is_weekend_stay',
                                                                    'total_guests',
                                                                    'has_children',
                                                                    'booking_intensity',
                                                                    'has_agent',
                                                                    'has_company'])]))])),
                ('model', LogisticRegression())])

### Тюн LogReg с помощью оптюны

In [3]:
!pip install optuna

In [4]:
log_reg_pipe.get_params()

{'memory': None,
 'steps': [('Data Preparation',
   Pipeline(steps=[('Feature Engineering', FeatureEngineer()),
                   ('feature encoding',
                    ColumnTransformer(transformers=[('categorial_oh',
                                                     OneHotEncoder(handle_unknown='ignore'),
                                                     ['hotel',
                                                      'arrival_date_month', 'meal',
                                                      'market_segment',
                                                      'distribution_channel',
                                                      'reserved_room_type',
                                                      'deposit_type',
                                                      'customer_type',
                                                      'country_group']),
                                                    ('numeric',
                                  

In [5]:
df = df.drop_duplicates()

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
import numpy as np
X = df.drop(columns='is_canceled')
y = df.is_canceled
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=22)
def objective(trial):
    max_iter = trial.suggest_int('model__max_iter', 1000, 10000, step=100)
    c = trial.suggest_float('model__C', 0.1, 10, log=True)
    solver = trial.suggest_categorical('model__solver', ['lbfgs', 'newton-cholesky', 'sag'])
    if solver == 'lbfgs' or solver=='newton-cholesky':
        penalty = 'l2'
    else:
        penalty = trial.suggest_categorical('model__penalty', [None, 'l1', 'l2'])
    log_reg_pipe.set_params(
        model__max_iter=max_iter,
        model__solver=solver,
        model__penalty=penalty,
        model__C=c,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
    scores = cross_val_score(
        log_reg_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [ ]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=10, n_jobs=-1)


[I 2026-06-16 16:24:19,294] A new study created in memory with name: no-name-1b662c7c-7b02-4157-89d8-50d71bebf8a1


  0%|          | 0/10 [00:00<?, ?it/s]

### Тюн RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_pipe = Pipeline(
    [('preparation', default_data_preparation_pipeline),
     ('model', RandomForestClassifier())]
)
def objective_rf(trial):
    # Гиперпараметры RandomForest
    n_estimators = trial.suggest_int('model__n_estimators', 50, 500, step=50)
    max_depth = trial.suggest_int('model__max_depth', 3, 20)       # можно заменить на suggest_categorical с None
    min_samples_split = trial.suggest_int('model__min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('model__min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('model__max_features', ['sqrt', 'log2', None])
    class_weight = trial.suggest_categorical('model__class_weight', ['balanced', None])
    bootstrap = trial.suggest_categorical('model__bootstrap', [True, False])

    # Устанавливаем параметры в пайплайн
    rf_pipe.set_params(
        model__n_estimators=n_estimators,
        model__max_depth=max_depth,
        model__min_samples_split=min_samples_split,
        model__min_samples_leaf=min_samples_leaf,
        model__max_features=max_features,
        model__class_weight=class_weight,
        model__bootstrap=bootstrap
    )

    # Кросс-валидация
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
    scores = cross_val_score(
        rf_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',          # метрика для классификации
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [ ]:

study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=10, n_jobs=-1)